# Qwen3-VL caption event generation (900-row systematic sample)

This notebook runs the caption-only Qwen3-VL-8B-Instruct preprocessing job for the SNU AI Challenge. It does not load the 38,140 train images. A small bundle supplies only `train.csv` and the event-generation code, while resumable state and the final `Id,Sentence,Events` CSV are written directly to Google Drive.

The model sees captions only. Images are deliberately excluded so the later CLIP image-event diagnostic does not reuse visual evidence during event construction.

In [13]:
import os
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

DRIVE_ROOT = Path('/content/drive/MyDrive/snu ai')
BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_qwen3vl_event_chunks_bundle.tar'
EXPECTED_BUNDLE_SHA256 = 'A026CE80D662E325959C17CA6CFBAF53107229226CB9DA38175C4588993E4A74'
WORKDIR = Path('/content/snu_ai')
TRAIN_CSV = WORKDIR / 'snuaichallenge_data/train.csv'
OUTPUT_DIR = DRIVE_ROOT / 'preprocessing/qwen3vl_event_chunks_900'
HF_HOME = Path('/content/huggingface')

SAMPLE_COUNT = 900
SEED = 42
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_REVISION = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'
QUANTIZATION = 'auto'  # full BF16/FP16 at >=22 GiB, otherwise 4-bit
BATCH_SIZE = 0         # 0 lets the script choose from detected VRAM
MAX_NEW_TOKENS = 192
MAX_RETRIES = 2

os.environ['HF_HOME'] = str(HF_HOME)
print('Drive output:', OUTPUT_DIR)


Drive output: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900


## 1. Mount Drive, install dependencies, and extract the small bundle

Upload `snu_colab_qwen3vl_event_chunks_bundle.tar` to `MyDrive/snu ai/` before running this cell. The archive is intentionally small because no images are needed.

In [14]:
import hashlib
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46',
], check=True)

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(
        f'Upload the small bundle to this exact path: {BUNDLE_PATH}'
    )
bundle_sha256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest().upper()
if bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        f'Bundle checksum mismatch: {bundle_sha256} != {EXPECTED_BUNDLE_SHA256}'
    )

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xf', str(BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
if not TRAIN_CSV.is_file():
    raise FileNotFoundError(TRAIN_CSV)
sys.path.insert(0, str(WORKDIR))
print('Bundle verified and extracted:', bundle_sha256)
print('train.csv:', TRAIN_CSV, TRAIN_CSV.stat().st_size, 'bytes')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bundle verified and extracted: A026CE80D662E325959C17CA6CFBAF53107229226CB9DA38175C4588993E4A74
train.csv: /content/snu_ai/snuaichallenge_data/train.csv 2136130 bytes


## 2. Prepare and verify the 900-caption systematic sample

This is CPU-only and does not load Qwen. Re-running it is safe: it deterministically reconstructs the same sample and preserves any compatible Drive state.

In [15]:
prepare_command = [
    sys.executable, '-m', 'preprocessing.qwen3vl_event_chunks',
    '--train-csv', str(TRAIN_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--sample-count', str(SAMPLE_COUNT),
    '--seed', str(SEED),
    '--model-id', MODEL_ID,
    '--revision', MODEL_REVISION,
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-retries', str(MAX_RETRIES),
    '--prepare-only',
]
subprocess.run(prepare_command, check=True, cwd=WORKDIR)

import csv
import json

sample_path = OUTPUT_DIR / 'sampled_captions.csv'
metadata_path = OUTPUT_DIR / 'metadata.json'
with sample_path.open('r', encoding='utf-8', newline='') as file:
    sampled_rows = list(csv.DictReader(file))
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
assert len(sampled_rows) == SAMPLE_COUNT
assert len({row['Id'] for row in sampled_rows}) == SAMPLE_COUNT
assert metadata['systematic_sampling']['sampled_no_ordering_true'] == 139
print('sample rows:', len(sampled_rows))
print('source-index span:',
      metadata['systematic_sampling']['first_source_index_zero_based'],
      'to', metadata['systematic_sampling']['last_source_index_zero_based'])
print('No_ordering=True rows:',
      metadata['systematic_sampling']['sampled_no_ordering_true'])
print('first Id:', sampled_rows[0]['Id'], 'last Id:', sampled_rows[-1]['Id'])


sample rows: 900
source-index span: 6 to 9531
No_ordering=True rows: 139
first Id: Qrt4Ax last Id: Sxk975


## 3. Inspect the inference GPU and selected loading mode

This is inference-only, not model training. The script uses full BF16/FP16 weights when total VRAM is at least 22 GiB; otherwise it selects NF4 4-bit loading. Override `QUANTIZATION` or `BATCH_SIZE` in the first cell if needed.

In [16]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
from preprocessing.qwen3vl_event_chunks import _gpu_configuration

gpu_config = _gpu_configuration(QUANTIZATION, BATCH_SIZE)
print('GPU:', gpu_config['name'])
print(f"VRAM: {gpu_config['total_vram_gib']:.2f} GiB")
print('compute capability:', gpu_config['compute_capability'])
print('dtype:', gpu_config['dtype_name'])
print('quantization:', gpu_config['quantization'])
print('generation batch size:', gpu_config['batch_size'])


GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GiB
compute capability: 8.0
dtype: bfloat16
quantization: none
generation batch size: 4


## 4. Run or resume Qwen event generation

Before loading Qwen, the script restores compatible rows from `state.jsonl` and promotes any structurally valid model outputs left in `errors.jsonl` by the older over-strict validator. If that reaches 900 rows, model loading and inference are skipped. Otherwise, the first run downloads the pinned model snapshot and only pending IDs are generated.

In [17]:
from collections import deque
from datetime import datetime, timezone

run_command = [
    sys.executable, '-m', 'preprocessing.qwen3vl_event_chunks',
    '--train-csv', str(TRAIN_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--sample-count', str(SAMPLE_COUNT),
    '--seed', str(SEED),
    '--model-id', MODEL_ID,
    '--revision', MODEL_REVISION,
    '--quantization', QUANTIZATION,
    '--batch-size', str(BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-retries', str(MAX_RETRIES),
]

state_path = OUTPUT_DIR / 'state.jsonl'
errors_path = OUTPUT_DIR / 'errors.jsonl'
fatal_error_path = OUTPUT_DIR / 'fatal_error.log'
log_path = OUTPUT_DIR / 'qwen_run.log'

def nonempty_line_count(path):
    if not path.is_file():
        return 0
    with path.open('r', encoding='utf-8') as file:
        return sum(bool(line.strip()) for line in file)

print('resuming valid state rows:', nonempty_line_count(state_path))
print('previous error records:', nonempty_line_count(errors_path))
print('full streaming log:', log_path)
print('fatal traceback log:', fatal_error_path)

tail = deque(maxlen=120)
with log_path.open('a', encoding='utf-8', buffering=1) as log_file:
    marker = f"\n===== run {datetime.now(timezone.utc).isoformat()} =====\n"
    log_file.write(marker)
    print(marker, end='')
    process = subprocess.Popen(
        run_command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors='replace',
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        tail.append(line)
    return_code = process.wait()

if return_code != 0:
    print('\n===== INNER ERROR TAIL =====')
    print(''.join(tail), end='')
    raise RuntimeError(
        f'Qwen event generation failed with exit code {return_code}. '
        f'The real traceback is printed above and saved at {log_path} '
        f'and {fatal_error_path}'
    )
print('Qwen event generation process completed successfully.')


resuming valid state rows: 900
previous error records: 632
full streaming log: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/qwen_run.log
fatal traceback log: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/fatal_error.log

===== run 2026-07-15T15:42:33.352781+00:00 =====
Wrote 900 rows to /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv; Qwen loading and inference were skipped.
Qwen event generation process completed successfully.


## 5. Validate the final file and inspect generated events

In [18]:
from collections import Counter

import pandas as pd

events_path = OUTPUT_DIR / 'events.csv'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
with events_path.open('r', encoding='utf-8', newline='') as file:
    reader = csv.DictReader(file)
    event_rows = list(reader)
    fieldnames = reader.fieldnames

assert metadata['status'] == 'complete', metadata
assert fieldnames == ['Id', 'Sentence', 'Events'], fieldnames
assert len(event_rows) == SAMPLE_COUNT
assert [row['Id'] for row in event_rows] == [row['Id'] for row in sampled_rows]

decoded = []
for row in event_rows:
    events = json.loads(row['Events'])
    assert isinstance(events, list) and 1 <= len(events) <= 4
    assert all(isinstance(event, str) and event.strip() for event in events)
    decoded.append({**row, 'DecodedEvents': events, 'EventCount': len(events)})

print('validated final rows:', len(decoded))
print('event-count distribution:', dict(sorted(Counter(
    row['EventCount'] for row in decoded
).items())))
print('final CSV:', events_path)
review = pd.DataFrame([
    {
        'Id': row['Id'],
        'Sentence': row['Sentence'],
        'Events': '\n'.join(
            f'{index + 1}. {event}'
            for index, event in enumerate(row['DecodedEvents'])
        ),
    }
    for row in decoded
]).sample(n=min(20, len(decoded)), random_state=SEED)
display(review)


validated final rows: 900
event-count distribution: {1: 154, 2: 447, 3: 211, 4: 88}
final CSV: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv


,Id,Sentence,Events
70,BmqcGY,"A hand adjusts a control panel, then a woman w...","1. A hand adjusts a control panel, then a woma..."
827,iN9SBo,A little girl is guided as she mounts a camel ...,1. A little girl is guided as she mounts a cam...
231,BdoK2L,"The person adjusts a yellow sock on their leg,...",1. The person adjusts a yellow sock on their l...
588,sGuQI8,The people then begin shaving the other person...,1. The people begin shaving the other person's...
39,gWyXHL,The scene transitions from a peeled potato and...,1. The scene transitions from a peeled potato ...
731,6uWrfu,The camera zooms out slightly as the guitarist...,1. The camera zooms out slightly as the guitar...
299,twjKu6,"The girl begins to twirl a baton, then moves i...",1. The girl begins to twirl a baton\n2. Then s...
110,MJNBT4,The girl in a pink hoodie stands at a sink as ...,1. The girl in a pink hoodie stands at a sink ...
72,XGrfcQ,The person crouches to inspect the bike's comp...,1. The person crouches to inspect the bike's c...
86,F84SDO,The person tightens the handlebars and lowers ...,1. The person tightens the handlebars and lowe...


## Output

The final file is saved at:

`/content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv`

Only this file is needed by the later CLIP diagnostic. Keep `state.jsonl` and `metadata.json` alongside it until all validation and manual review are finished.